# Module 7: Agentic Workflows

_Last verified: 2026-09-27, against LangGraph 1.2_

**Goal:** Build progressively complex agent systems — from a single tool-calling loop up to a multi-agent LangGraph workflow with a human approval step.

Every model call goes through the course's provider layer (`llm/`), so any model in `llm/models.json` works and the registry decides what each model is sent. LangGraph 1.x does not need LangChain's chat-model wrappers: a node is a plain function, and here it calls `llm.complete()`.

**What you'll learn:**
1. What makes an LLM an "agent" (tool use + decision loop)
2. Defining tools as a name, a description and a JSON schema
3. The agent loop by hand, as [Spine 1](../../spine/01-loop/README.md#the-loop) builds it
4. The same loop as a LangGraph 1.x `StateGraph`
5. Multi-agent routing (Researcher / Coder / Advisor)
6. Human-in-the-loop with `interrupt()` and a checkpointer

The [agent frameworks page](README.md) maps each of these onto LangGraph, the OpenAI Agents SDK and Google ADK 2.0.

---

## 0. Setup

In [ ]:
%pip install -q -r requirements.txt

In [ ]:
import sys
sys.path.insert(0, "../..")   # the provider layer (llm/) lives at the repo root
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env")

from llm import Message, ToolResult, ToolSpec, complete, configured_registry, default_model
from llm.transport import HttpTransport

REGISTRY = configured_registry()               # the registry, plus a local server if you set one up
MODEL = default_model(REGISTRY)                # LLM_MODEL, or the cheapest model your key reaches
TRANSPORT = HttpTransport(provider=REGISTRY.get(MODEL).provider)

def call_model(messages, tools=(), system=None):
    """One model call through the provider layer. Returns a normalized Response."""
    return complete(model=MODEL, messages=messages, tools=tools, system=system,
                    transport=TRANSPORT, registry=REGISTRY)

print(f"✅ Ready — model: {MODEL}")

---
## 1. What Is an Agent?

A plain LLM call is **stateless**: question → answer, done.

An **agent** adds a loop:

```
User goal
   │
   ▼
[LLM] ──thinks──▶ "I need to call tool X"
   │                     │
   │◀── tool result ──────┘
   │
[LLM] ──thinks──▶ "I need tool Y too" → repeat
   │
[LLM] ──thinks──▶ "I have enough info now"
   │
Final answer
```

The LLM decides **which** tools to call, **when** to stop, and **how** to combine results. That decision-making capacity is what makes it an agent.

---
## 2. Defining Tools

A tool is a name, a description the model routes on, and a JSON schema for its arguments: an `llm.ToolSpec`. The provider layer translates it into each provider's own tool format. The Python function that runs it stays yours.

In [ ]:
import math, json, datetime

def calculator(expression: str) -> str:
    try:
        return str(eval(expression, {"__builtins__": {}}, {"math": math}))
    except Exception as e:
        return f"Error: {e}"

def get_current_date(timezone: str = "UTC") -> str:
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S") + f" ({timezone})"

def word_count(text: str) -> str:
    words = len(text.split())
    sentences = text.count('.') + text.count('!') + text.count('?')
    return json.dumps({"words": words, "sentences": sentences, "characters": len(text)})

def search_knowledge_base(query: str) -> str:
    # Simulated KB — in production this would hit a real retriever
    kb = {
        "rag": "RAG combines retrieval with generation. It retrieves relevant documents and uses them as context.",
        "lora": "LoRA fine-tunes models by adding small trainable matrices, reducing parameters by 99%+.",
        "agent": "Agents use LLMs as reasoning engines that decide which tools to call and when to stop.",
        "evaluation": "LLM evaluation uses automated metrics (ROUGE), LLM-as-judge, and human review.",
    }
    results = [v for k, v in kb.items() if k in query.lower()]
    return results[0] if results else "No relevant information found in knowledge base."

def _string_arg(name, description):
    return {"type": "object", "properties": {name: {"type": "string", "description": description}}, "required": [name]}

TOOL_SPECS = [
    ToolSpec("calculator",
             "Evaluate a mathematical expression. Input must be a valid Python math expression, "
             "e.g. '2 + 2', 'math.sqrt(144)', '(10 * 5) / 2'.",
             _string_arg("expression", "A Python math expression")),
    ToolSpec("get_current_date", "Get the current date and time.",
             {"type": "object", "properties": {"timezone": {"type": "string", "default": "UTC"}}}),
    ToolSpec("word_count", "Count words, sentences, and characters in a text.",
             _string_arg("text", "The text to count")),
    ToolSpec("search_knowledge_base",
             "Search the LLM engineering knowledge base (RAG, LoRA, agents, evaluation). Returns a snippet.",
             _string_arg("query", "What to look up")),
]
TOOLS = {"calculator": calculator, "get_current_date": get_current_date,
         "word_count": word_count, "search_knowledge_base": search_knowledge_base}

def run_tool(call) -> ToolResult:
    """Run one ToolCall the model asked for and wrap the result to send back."""
    fn = TOOLS.get(call.name)
    if fn is None:
        return ToolResult(call.id, call.name, f"Unknown tool: {call.name}", is_error=True)
    return ToolResult(call.id, call.name, fn(**call.arguments))

print(f"Defined {len(TOOL_SPECS)} tools:")
for spec in TOOL_SPECS:
    print(f"  - {spec.name}: {spec.description[:60]}...")

---
## 3. The Agent Loop, by Hand

The model alternates between reasoning ("I should calculate X") and acting (asking for a tool call). The loop runs the tools, sends the results back, and stops when the model answers — or when it runs out of steps. This is the loop [Spine 1](../../spine/01-loop/README.md#the-loop) builds for the Flagship Agent.

In [ ]:
SYSTEM = "You are a helpful assistant with access to tools. Use them when needed to answer accurately."

def run_agent(question: str, step_limit: int = 5) -> str:
    messages = [Message.user(question)]
    for _ in range(step_limit):
        response = call_model(messages, tools=TOOL_SPECS, system=SYSTEM)
        messages.append(Message.assistant(response.text or None, response.tool_calls))
        if not response.tool_calls:          # stop condition: the model answered
            return response.text
        for call in response.tool_calls:
            print(f"  → {call.name}({dict(call.arguments)})")
        messages.append(Message.tool([run_tool(call) for call in response.tool_calls]))
    return f"Stopped: no answer within {step_limit} steps."   # stop condition: the step limit

print("Final Answer:", run_agent("What is the square root of 1764, and what is today's date?"))

In [ ]:
# Multi-step reasoning: requires combining tools
print("Final Answer:", run_agent(
    "Count the words in this sentence: 'The quick brown fox jumps over the lazy dog.' "
    "Then tell me what RAG stands for."
))

---
## 4. The Same Loop as a LangGraph State Machine

**LangGraph** makes the loop's flow explicit: you define nodes (functions) and edges (routing rules) as a graph. The state here is a list of the provider layer's `Message`s; LangGraph appends each node's return value to it.

Benefits: inspectable, testable, supports cycles, checkpoints and human interrupts. The step limit becomes LangGraph's `recursion_limit`.

In [ ]:
import operator
from typing import Annotated, Literal, TypedDict
from langgraph.graph import END, START, StateGraph

# 1. Shared state: an append-only transcript
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

# 2. Nodes are plain functions over the state
def model_node(state: AgentState) -> dict:
    """The model decides: answer directly or call a tool."""
    response = call_model(state["messages"], tools=TOOL_SPECS, system=SYSTEM)
    return {"messages": [Message.assistant(response.text or None, response.tool_calls)]}

def tools_node(state: AgentState) -> dict:
    """Run every tool call the model asked for."""
    calls = state["messages"][-1].tool_calls
    return {"messages": [Message.tool([run_tool(call) for call in calls])]}

# 3. Routing: loop back through the tools, or stop
def should_continue(state: AgentState) -> Literal["tools", "end"]:
    return "tools" if state["messages"][-1].tool_calls else "end"

# 4. Build the graph
graph = StateGraph(AgentState)
graph.add_node("model", model_node)
graph.add_node("tools", tools_node)
graph.add_edge(START, "model")
graph.add_conditional_edges("model", should_continue, {"tools": "tools", "end": END})
graph.add_edge("tools", "model")

app = graph.compile()
print("✅ Graph compiled")
print("Nodes:", list(app.get_graph().nodes.keys()))

In [ ]:
# Run the graph; recursion_limit caps the node steps, like the loop's step limit
result = app.invoke({"messages": [Message.user("What is 15% of 840? Show your calculation.")]},
                    config={"recursion_limit": 10})

print("Message trace:")
for msg in result["messages"]:
    if msg.tool_results:
        content = f"[tool results: {[r.content[:40] for r in msg.tool_results]}]"
    elif msg.tool_calls:
        content = f"[tool call: {[c.name for c in msg.tool_calls]}]"
    else:
        content = msg.text
    print(f"  {msg.role:10s}: {str(content)[:100]}")

---
## 5. Multi-Agent Routing

Different tasks need different specialists. A **router** inspects the request and dispatches to the right agent.

In [ ]:
def make_specialist(role: str, expertise: str):
    """Factory: create a specialist agent node."""
    system = f"You are a {role}. {expertise} Be concise (2-3 sentences)."
    def node(state: AgentState) -> dict:
        print(f"  → Routing to: {role}")
        response = call_model(state["messages"], system=system)
        return {"messages": [Message.assistant(f"[{role}] {response.text}")]}
    node.__name__ = role.lower().replace(" ", "_")
    return node

researcher = make_specialist("Research Analyst", "Explain concepts clearly with facts and examples.")
coder      = make_specialist("Senior Engineer",  "Give precise technical answers with code snippets.")
advisor    = make_specialist("Strategy Advisor", "Give practical, actionable business advice.")

def router(state: AgentState) -> Literal["researcher", "coder", "advisor"]:
    """Classify the query and route to the right specialist."""
    question = state["messages"][-1].text
    route_prompt = ("Classify this question into exactly one category: 'research', 'code', or 'strategy'.\n"
                    f"Output only the single word.\nQuestion: {question}")
    classification = call_model([Message.user(route_prompt)]).text.strip().lower()
    mapping = {"research": "researcher", "code": "coder", "strategy": "advisor"}
    return mapping.get(classification, "researcher")

# Build multi-agent graph
multi = StateGraph(AgentState)
multi.add_node("researcher", researcher)
multi.add_node("coder",      coder)
multi.add_node("advisor",    advisor)
multi.add_conditional_edges(START, router, {"researcher": "researcher", "coder": "coder", "advisor": "advisor"})
multi.add_edge("researcher", END)
multi.add_edge("coder",      END)
multi.add_edge("advisor",    END)
multi_app = multi.compile()

test_questions = [
    "What is transformer attention and how does it work?",
    "Write a Python function to chunk text into overlapping windows.",
    "Should I fine-tune my own model or use an API for a customer support bot?",
]

for q in test_questions:
    print(f"\nQ: {q}")
    result = multi_app.invoke({"messages": [Message.user(q)]})
    print(f"A: {result['messages'][-1].text[:200]}...")

---
## 6. Human-in-the-Loop

Some decisions are too consequential for full automation. LangGraph's `interrupt()` pauses the graph inside a node and hands a value to your application; a checkpointer saves the paused state under a `thread_id`. You resume by invoking the graph again with `Command(resume=...)`, and `interrupt()` returns the human's answer.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

class ReviewState(TypedDict):
    question: str
    draft: str
    approved: bool

def draft_node(state: ReviewState) -> dict:
    """The model drafts a response."""
    draft = call_model([Message.user(f"Draft a short response to: {state['question']}")]).text
    print(f"\n📝 Draft ready: {draft[:150]}...")
    return {"draft": draft}

def review_node(state: ReviewState) -> dict:
    """Pause here until a human approves or rejects the draft."""
    decision = interrupt({"draft": state["draft"], "ask": "Approve this draft? (yes/no)"})
    return {"approved": str(decision).strip().lower() in ("yes", "y", "approve")}

def finalize_node(state: ReviewState) -> dict:
    return {"draft": state["draft"] if state["approved"] else "Draft rejected. Please rephrase your question."}

hitl = StateGraph(ReviewState)
hitl.add_node("draft",    draft_node)
hitl.add_node("review",   review_node)
hitl.add_node("finalize", finalize_node)
hitl.add_edge(START,      "draft")
hitl.add_edge("draft",    "review")
hitl.add_edge("review",   "finalize")
hitl.add_edge("finalize", END)
hitl_app = hitl.compile(checkpointer=InMemorySaver())   # interrupts need a checkpointer

config = {"configurable": {"thread_id": "review-session-1"}}
paused = hitl_app.invoke({"question": "Explain quantum entanglement", "draft": "", "approved": False}, config=config)
print("\n⏸️  Paused for review:", paused["__interrupt__"][0].value["ask"])

# In a real app a person answers in a UI; here we approve in code
result = hitl_app.invoke(Command(resume="yes"), config=config)
print(f"\n👤 Approved: {result['approved']}")
print(f"Final: {result['draft'][:200]}")

---
## 7. Agent Design Patterns Cheat Sheet

| Pattern | When to use | Trade-off |
|---------|-------------|----------|
| **Hand-written loop** | General tool use; you want to read every step | You own stop conditions and state |
| **LangGraph** | Complex flows, loops, human review, checkpoints | More setup, fully inspectable |
| **Multi-agent router** | Diverse task types | Routing overhead, need good classifier |
| **Hierarchical** | Large projects (manager → sub-agents) | High token cost |
| **Parallel agents** | Independent sub-tasks | Need result aggregation |

## 🧪 Exercises

1. **Add a tool**: Add a `unit_converter` tool (e.g., km↔miles, °C↔°F). Verify the agent uses it automatically when asked a conversion question.
2. **Tool failure handling**: Make `calculator` return `is_error=True` for division by zero. What does the agent do? How can you make it recover gracefully?
3. **Improve the router**: The current router uses an LLM classification call. Replace it with keyword matching. Which approach is faster? Which is more accurate?
4. **Cycle limit**: Ask the graph agent a question it can't answer with available tools (e.g., "What's the weather in Tokyo?"). Lower `recursion_limit` until it raises `GraphRecursionError`. How would you turn that into a clean "out of steps" answer?
5. **Reject the draft**: Resume the human-in-the-loop graph with `Command(resume="no")`. Then change `finalize_node` to route back to `draft` with the reviewer's feedback.

---
**Next:** [Module 08 — LLMOps & Observability](../observability/llmops_observability.ipynb)

---
## 🧪 Exercises

1. **Single Agent**: Build agent with 3 tools. How does it decide which to use?
2. **Multi-Agent Debate**: Two agents with opposing views debate for 5 rounds.
3. **LangGraph**: Build workflow with 3 nodes, 2 conditional edges, and a retry loop.